# **PROYECTO RAG**

## **Intalación de librerias**

In [1]:
!pip install -q torch --upgrade
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community
!pip install -q sentence-transformers faiss-cpu
!pip install -q pypdf beautifulsoup4 requests
!pip install pypdf
!pip install rouge-score
!pip install rank-bm25
!pip install nltk

In [2]:
from pypdf import PdfReader
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from rank_bm25 import BM25Okapi
import faiss
import numpy as np
import requests
import re
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import textwrap

## **Ingesta de Data**

*Paginas Web*

In [3]:
urls = {
    "https://es.wikipedia.org/wiki/Aprendizaje_autom%C3%A1tico",
    "https://www.ibm.com/es-es/think/topics/supervised-learning",
    "https://www.ibm.com/es-es/think/topics/deep-learning",
    "https://es.wikipedia.org/wiki/Aprendizaje_profundo"
}

web_documents = []

for url in urls:

    loader = WebBaseLoader(url)

    docs = loader.load()

    web_documents.extend(docs)

print("Total web documents:", len(web_documents))

Total web documents: 4


*Libros*

In [4]:
#Esta función nos ayudara a limpiar la data
def clean_text(text):

    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

#Con esta función se podrá detectar fórmulas
def tag_formulas(text):

    if "=" in text and ("+" in text or "∑" in text):
        return "[FORMULA] " + text

    return text

#Funcion para detectar figuras
def tag_figures(text):

    if "figure" in text.lower():
        return "[FIGURE] " + text

    return text

In [5]:

pdf_urls = [
"https://github.com/AnghieLizzeth22/IA_GEN_PROYECTO_RAG_ML/raw/79eef83bb39f0bca9eb8ac223c173deb87b0cc03/docs/Machine-Learning-for-Absolute-Beginners.pdf",
"https://github.com/AnghieLizzeth22/IA_GEN_PROYECTO_RAG_ML/raw/79eef83bb39f0bca9eb8ac223c173deb87b0cc03/docs/ML%20Math.pdf",
"https://github.com/AnghieLizzeth22/IA_GEN_PROYECTO_RAG_ML/raw/79eef83bb39f0bca9eb8ac223c173deb87b0cc03/docs/Deep%20Learning%20by%20Ian%20Goodfellow%2C%20Yoshua%20Bengio%2C%20Aaron%20Courville.pdf"
]

pdf_files = []

for i, url in enumerate(pdf_urls):

    filename = f"book_{i}.pdf"

    response = requests.get(url)

    with open(filename, "wb") as f:
        f.write(response.content)

    pdf_files.append(filename)

print("PDF descargados:", pdf_files)


pdf_documents = []

for file in pdf_files:

    loader = PyPDFLoader(file)

    docs = loader.load()

    pdf_documents.extend(docs)

print("Total páginas PDF:", len(pdf_documents))

for doc in pdf_documents:

    doc.metadata["source"] = doc.metadata.get("source", "ML_book")
    doc.metadata["page"] = doc.metadata.get("page", "unknown")

    text = doc.page_content

    text = clean_text(text)
    text = tag_formulas(text)
    text = tag_figures(text)

    doc.page_content = text


PDF descargados: ['book_0.pdf', 'book_1.pdf', 'book_2.pdf']
Total páginas PDF: 1397


In [6]:
print(pdf_documents[10].page_content[:500])

[FIGURE] improve performance based on data and empirical information; all without direct programming commands. This is what Arthur Samuel described as the ability to learn without being explicitly programmed. Samuel didn’t infer that machines may formulate decisions with no upfront programming. On the contrary, machine learning is heavily dependent on code input. Instead, he observed machines can perform a set task using input data rather than relying on a direct input command . Figure 2: Compar


Unimos todos los documentos cargados

In [7]:
documents = web_documents + pdf_documents

print("Total documentos:", len(documents))

Total documentos: 1401


## **Chunking + Embeddings**

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 5187


Se convierte el texto a vectores, conocido como embeddings

In [9]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [doc.page_content for doc in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/163 [00:00<?, ?it/s]

## **FAISS con HNSW**

In [10]:
print(embeddings.shape)

(5187, 384)


In [11]:
dimension = embeddings.shape[1]

index = faiss.IndexHNSWFlat(dimension, 32)

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 50

index.add(embeddings)

print("Vectores indexados:", index.ntotal)

Vectores indexados: 5187


## **Retrieval - Hybrid Retrieval**

In [12]:
def retrieve(query, k=10):

    q_embedding = embedding_model.encode([query],
        normalize_embeddings=True)

    q_embedding = np.array(q_embedding).astype("float32")

    distances, indices = index.search(q_embedding, k)

    results = [chunks[i] for i in indices[0]]

    return results

***BM25***

In [13]:
tokenized_chunks = [doc.page_content.lower().split()
    for doc in chunks]

bm25 = BM25Okapi(tokenized_chunks)

In [14]:
def bm25_search(query, k=5):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:k]

    results = [chunks[i] for i in top_indices]

    return results

***Hybrid Retrieval***

In [15]:
def hybrid_retrieve(query, k=10):

    # vector search
    vector_results = retrieve(query, k)

    # keyword search
    keyword_results = bm25_search(query, k)

    # combinar resultados
    combined = vector_results + keyword_results

    # eliminar duplicados
    unique_docs = list(
        {doc.page_content: doc for doc in combined}.values()
    )

    return unique_docs[:k]

## **Reranker cross-encoder**

In [16]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank(query, docs, top_k=5):

    if not docs:
        return []

    pairs = [(query, doc.page_content) for doc in docs]

    scores = reranker.predict(
        pairs,
        batch_size=16
    )

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for score, doc in ranked[:top_k]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Compresion por contexto

In [17]:
def compress_context(query, docs, embed_model, top_sentences=3):

    query_emb = embed_model.encode([query])

    compressed_docs = []

    for doc in docs:

        paragraphs = doc.page_content.split("\n")

        para_emb = embed_model.encode(paragraphs)

        scores = cosine_similarity(query_emb, para_emb)[0]

        ranked = sorted(
            zip(scores, paragraphs),
            reverse=True
        )

        best = [p for _, p in ranked[:top_sentences]]

        compressed_docs.append("\n".join(best))

    return compressed_docs

## **Pipeline RAG**

***LLM Generation LLama2***

In [18]:
model_name = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.1
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [19]:
def rag_pipeline(question):

    retrieved_docs = hybrid_retrieve(question)

    reranked_docs = rerank(question, retrieved_docs)

    compressed = compress_context(question, reranked_docs, embedding_model)

    context = "\n\n".join(compressed[:3])

    prompt = f"""
Eres un experto en Machine Learning.
Responde en español usando definiciones claras y precisas.
Cada oración debe basarse únicamente en la información del contexto proporcionado.
No agregues información que no esté en el contexto.
Si la información no está en el contexto, di que no está disponible.

Context:
{context}

Question:
{question}

Answer:
"""

    #response = generator(prompt)[0]["generated_text"]
    output = generator(
        prompt,
        max_new_tokens=200,
        temperature=0.1
    )[0]["generated_text"]

    response = output.split("Answer:")[-1].strip()

#Si se quiere ver el Log descomenta lo siguiente:
#print(f"Retrieved: {len(retrieved_docs)} docs")
#print(f"After rerank: {len(reranked_docs)} docs")
#print(f"Compressed paragraphs: {len(compressed)}")

    return response, reranked_docs

## **Citation per sentence**

In [20]:
#Funcion para la lista de fuentes
def answer_with_citations(question):

    answer, docs = rag_pipeline(question)

    sources = []

    for i, doc in enumerate(docs):

        src = doc.metadata.get("source", "unknown")

        sources.append(f"[{i+1}] {src}")

    return answer, sources

In [21]:
#Función citation per sentence , es decir estariamos citando en cada oracion
def answer_with_citations(answer, docs, embedding_model):

    sentences = re.split(r'(?<=[.!?]) +', answer)

    doc_texts = [doc.page_content for doc in docs]
    doc_embeddings = embedding_model.encode(doc_texts)

    citations = []

    for sent in sentences:

        sent_emb = embedding_model.encode([sent])[0]

        scores = cosine_similarity([sent_emb], doc_embeddings)[0]

        best_idx = scores.argmax()
        best_score = scores[best_idx]

        citations.append((sent, best_idx, best_score))

    return citations

## **Hallucination guard + grounding**

In [22]:
def grounding_score(answer, context):

    a_emb = embedding_model.encode([answer])
    c_emb = embedding_model.encode([context])

    score = cosine_similarity(a_emb, c_emb)[0][0]

    return float(score)

In [23]:
def grounding_score_det(answer, docs):

    answer_emb = embedding_model.encode([answer])[0]

    scores = []

    for doc in docs:
        chunk_emb = embedding_model.encode([doc.page_content])[0]
        score = cosine_similarity([answer_emb], [chunk_emb])[0][0]
        scores.append(score)

    return max(scores)

## **Evaluación automática**

*Rouge*

In [24]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def evaluate(reference, prediction):

    score = scorer.score(reference, prediction)

    return score["rougeL"].fmeasure

*Bleu*

In [25]:
smooth = SmoothingFunction().method1
def bleu_score(reference, answer):

    ref_tokens = reference.split()
    ans_tokens = answer.split()

    score = sentence_bleu([ref_tokens], ans_tokens, smoothing_function=smooth)

    return score

## **DEMO**

Escribes tu pregunta relacionada al área del Machine Learning

In [26]:
#question = "Explica la diferencia entre aprendizaje supervisado y no supervisado"
questions = [
    "¿Qué es overfitting?",
    "¿Qué es el aprendizaje supervisado?",
    "¿Qué es el aprendizaje no supervisado?",
    "¿Qué es la regresión lineal?",
    "¿Qué es un árbol de decisión?",
    "¿Qué es el descenso por gradiente?"
]


references = {

"¿Qué es overfitting?":
"El overfitting ocurre cuando un modelo se ajusta demasiado a los datos de entrenamiento y no generaliza bien a nuevos datos. Esto sucede cuando el modelo aprende patrones específicos del entrenamiento en lugar de relaciones generales.",

"¿Qué es el aprendizaje supervisado?":
"El aprendizaje supervisado es una técnica de machine learning que utiliza datos etiquetados para entrenar modelos. El objetivo es aprender patrones que permitan predecir resultados en nuevos datos.",

"¿Qué es el aprendizaje no supervisado?":
"El aprendizaje no supervisado utiliza datos no etiquetados para descubrir patrones o relaciones en los datos. El modelo aprende estructuras ocultas sin conocer previamente las respuestas correctas.",

"¿Qué es la regresión lineal?":
"La regresión lineal es un método que modela la relación entre una variable dependiente y una o más variables independientes. Se utiliza para realizar predicciones sobre valores continuos.",

"¿Qué es un árbol de decisión?":
"Un árbol de decisión es un modelo predictivo que utiliza una estructura de árbol para tomar decisiones. Cada nodo representa una condición sobre los datos y las ramas conducen a diferentes resultados.",

"¿Qué es el descenso por gradiente?":
"El descenso por gradiente es un algoritmo de optimización utilizado para minimizar una función de pérdida. Ajusta iterativamente los parámetros del modelo en la dirección que reduce el error."
}

In [27]:
rouge_scores = []
grounding_scores = []
bleu_scores = []

for q in questions:

    print("\n===============================")
    print("Pregunta:", q)

    # ejecutar pipeline
    answer, docs = rag_pipeline(q)

    reference = references[q]

    # métricas
    rouge = evaluate(reference, answer)
    bleu = bleu_score(reference, answer)
    grounding = grounding_score_det(answer, docs)

    rouge_scores.append(rouge)
    bleu_scores.append(bleu)
    grounding_scores.append(grounding)

    print("\nRespuesta:")
    print(textwrap.fill(answer, width=100))

    print("\nCitation per sentence:")

    citations = answer_with_citations(answer, docs, embedding_model)

    for sent, src, score in citations:

        source_name = docs[src].metadata.get("source", f"doc_{src+1}")

        wrapped_sent = textwrap.fill(sent, width=100)
        print(f"- {wrapped_sent} [{src+1} | sim={score:.2f} | {source_name}]")

    print("\nFuentes:")

    for i, doc in enumerate(docs):

        source_name = doc.metadata.get("source", f"doc_{i+1}")

        print(f"[{i+1}] {source_name}")

    print("\nROUGE-L:", round(rouge, 3))
    print("BLEU:", round(bleu,3))
    print("Grounding:", round(grounding, 3))




Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Pregunta: ¿Qué es overfitting?


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Respuesta:
Para prevenir el overfitting, se pueden utilizar técnicas como el regularización, el agregado de
información adicional, el uso de modelos más complejos o el uso de técnicas de divulgación. También
se puede probar el modelo con datos diferentes de los datos de entrenamiento para ver si se ajusta
demasiado a los datos de entrenamiento.

Citation per sentence:
- Para prevenir el overfitting, se pueden utilizar técnicas como el regularización, el agregado de
información adicional, el uso de modelos más complejos o el uso de técnicas de divulgación. [1 | sim=0.55 | https://www.ibm.com/es-es/think/topics/supervised-learning]
- También se puede probar el modelo con datos diferentes de los datos de entrenamiento para ver si se
ajusta demasiado a los datos de entrenamiento. [2 | sim=0.68 | https://www.ibm.com/es-es/think/topics/supervised-learning]

Fuentes:
[1] https://www.ibm.com/es-es/think/topics/supervised-learning
[2] https://www.ibm.com/es-es/think/topics/supervised-learning


Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Respuesta:
El aprendizaje no supervisado es un método de aprendizaje automático en el que el modelo utiliza
datos no etiquetados para descubrir patrones y relaciones en los datos. El modelo no tiene
información objetiva que guíe su aprendizaje, por lo que debe descubrir por sí mismo qué patrones y
relaciones hay en los datos.

Citation per sentence:
- El aprendizaje no supervisado es un método de aprendizaje automático en el que el modelo utiliza
datos no etiquetados para descubrir patrones y relaciones en los datos. [1 | sim=0.77 | https://www.ibm.com/es-es/think/topics/supervised-learning]
- El modelo no tiene información objetiva que guíe su aprendizaje, por lo que debe descubrir por sí
mismo qué patrones y relaciones hay en los datos. [3 | sim=0.63 | https://es.wikipedia.org/wiki/Aprendizaje_profundo]

Fuentes:
[1] https://www.ibm.com/es-es/think/topics/supervised-learning
[2] https://es.wikipedia.org/wiki/Aprendizaje_autom%C3%A1tico
[3] https://es.wikipedia.org/wiki/Aprendizaje_p

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Respuesta:
La regresión lineal es una técnica de aprendizaje automático utilizada para identificar la relación
entre una variable dependiente continua y una o más variables independientes. La regresión lineal se
utiliza para hacer predicciones sobre resultados futuros y se expresa a través de una línea recta.

Citation per sentence:
- La regresión lineal es una técnica de aprendizaje automático utilizada para identificar la relación
entre una variable dependiente continua y una o más variables independientes. [1 | sim=0.72 | https://www.ibm.com/es-es/think/topics/supervised-learning]
- La regresión lineal se utiliza para hacer predicciones sobre resultados futuros y se expresa a
través de una línea recta. [1 | sim=0.77 | https://www.ibm.com/es-es/think/topics/supervised-learning]

Fuentes:
[1] https://www.ibm.com/es-es/think/topics/supervised-learning
[2] https://www.ibm.com/es-es/think/topics/supervised-learning
[3] https://www.ibm.com/es-es/think/topics/supervised-learning
[4] https

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Respuesta:
El descenso por gradiente es un algoritmo de optimización que se utiliza para encontrar los ajustes
de parámetros que minimizan una función de pérdida. El algoritmo funciona "desciendiendo" el
gradiente de la función de pérdida, es decir, reduciendo la pérdida en cada iteración. El objetivo
del algoritmo es encontrar los ajustes de parámetros que minimizan la función de pérdida, lo que se
conoce como el gradiente mínimo.

Citation per sentence:
- El descenso por gradiente es un algoritmo de optimización que se utiliza para encontrar los ajustes
de parámetros que minimizan una función de pérdida. [1 | sim=0.70 | https://www.ibm.com/es-es/think/topics/deep-learning]
- El algoritmo funciona "desciendiendo" el gradiente de la función de pérdida, es decir, reduciendo la
pérdida en cada iteración. [1 | sim=0.71 | https://www.ibm.com/es-es/think/topics/deep-learning]
- El objetivo del algoritmo es encontrar los ajustes de parámetros que minimizan la función de
pérdida, lo que se c

In [28]:
print("\n===============================")
print("RESULTADOS PROMEDIO")

print("ROUGE promedio:", round(np.mean(rouge_scores), 2))
print("BLEU promedio:", round(np.mean(bleu_scores),2))
print("Grounding promedio:", round(np.mean(grounding_scores), 2))


RESULTADOS PROMEDIO
ROUGE promedio: 0.34
BLEU promedio: 0.15
Grounding promedio: 0.78


El grounding score nos arroja una puntuación mayor a 0.8, entonces se considera una respuesta muy bien respaldada.

In [29]:
# !pip freeze > requirements.txt